In [2]:
%run "scripts\load_daily_variables.py"

Loading Variables: 100%|██████████| 26/26 [01:35<00:00,  3.67s/it]


In [4]:
import numpy as np
from joblib import Parallel, delayed

SIGNAL_COLUMN = "regime_crossing_ma_short_2050200"


def _collect_symbol_occurrences(symbol, df, signal_column):
    """Collect contiguous -1 runs for one symbol in a worker process."""
    if signal_column not in df.columns:
        return symbol, [], True

    is_short = df[signal_column].eq(-1).to_numpy()
    starts = np.flatnonzero(is_short & np.r_[True, ~is_short[:-1]])
    stops = np.flatnonzero(is_short & np.r_[~is_short[1:], True]) + 1
    runs = [df.iloc[start:stop].copy() for start, stop in zip(starts, stops)]
    return symbol, runs, False


def collect_ma_short_occurrences(symbols, signal_column=SIGNAL_COLUMN, n_jobs=-1):
    """Return one DataFrame per contiguous -1 run, using multiple processes."""
    tasks = (
        delayed(_collect_symbol_occurrences)(symbol, symbol_data.df, signal_column)
        for symbol, symbol_data in symbols.items()
    )
    results = Parallel(
        n_jobs=n_jobs,
        prefer="processes",
        return_as="generator",
    )(tasks)

    occurrences = {}
    missing_column = []
    for symbol, runs, is_missing in tqdm(
        results,
        total=len(symbols),
        desc="Collecting MA-short occurrences",
    ):
        occurrences[symbol] = runs
        if is_missing:
            missing_column.append(symbol)

    total_samples = sum(len(samples) for samples in occurrences.values())
    symbols_with_samples = sum(bool(samples) for samples in occurrences.values())
    print(
        f"Collected {total_samples:,} contiguous samples from "
        f"{symbols_with_samples:,} of {len(occurrences):,} symbols."
    )
    if missing_column:
        print(f"Skipped {len(missing_column):,} symbols missing {signal_column!r}.")

    return occurrences


ma_short_occurrences = collect_ma_short_occurrences(symbols)

Collected 52,654 contiguous samples from 2,838 of 3,021 symbols.


In [10]:
ma_short_occurrences.keys()

dict_keys(['PLTR', 'ORCL', 'WDC', 'INTC', 'AMAT', 'GOOGL', 'MU', 'GOOG', 'AAPL', 'LRCX', 'NVDA', 'TSM', 'META', 'MSFT', 'AVGO', 'SNDK', 'AMD', 'TSLA', 'AMZN', 'MRVL', 'APP', 'CRM', 'WMT', 'CRWV', 'UNH', 'STX', 'ARM', 'DELL', 'LITE', 'JPM', 'NBIS', 'AAL', 'GEV', 'HOOD', 'BE', 'CRDO', 'RKLB', 'QCOM', 'CSCO', 'NOW', 'V', 'LLY', 'COST', 'ALAB', 'NFLX', 'MSTR', 'CRWD', 'TXN', 'PANW', 'JNJ', 'GLW', 'IBM', 'KLAC', 'ASML', 'GS', 'VRT', 'XOM', 'ABBV', 'ADBE', 'IREN', 'CAT', 'ASTS', 'COHR', 'CVX', 'BAC', 'ANET', 'SNOW', 'SMCI', 'ADI', 'BKNG', 'TER', 'ACN', 'MA', 'UBER', 'BABA', 'GE', 'SOFI', 'C', 'INTU', 'VZ', 'HD', 'MPWR', 'AZO', 'AAOI', 'T', 'ON', 'HON', 'PG', 'ABT', 'MCD', 'KO', 'MRK', 'ECHO', 'CIEN', 'BA', 'LIN', 'COIN', 'HPE', 'DDOG', 'NKE', 'WFC', 'APH', 'NXPI', 'TMUS', 'NEE', 'SCHW', 'SPGI', 'AXP', 'DASH', 'MS', 'AMGN', 'FTNT', 'ISRG', 'DIS', 'CRCL', 'MCHP', 'PEP', 'VRTX', 'ROKU', 'TMO', 'RTX', 'GILD', 'CEG', 'SHOP', 'FCX', 'CME', 'PFE', 'FLEX', 'PWR', 'PM', 'NOK', 'IONQ', 'ETN', 'TJX', '

In [12]:
ma_short_occurrences['APP'][-1]

,Open,High,Low,Close,Volume,VWAP,5DMA,10DMA,20DMA,50DMA,200DMA,AvgV20,RVol,Dollar_Volume,AvgDV20,TR,ATR_14,ATRs_from_5DMA,ATRs_from_10DMA,ATRs_from_20DMA,ATRs_from_50DMA,ATRs_from_200DMA,ATR_14_signal,RSI_14,RSI_14_signal,RSI_14_diff,n_high,n_low,%k,%d,%k_diff,+DI,-DI,ADX,ADX_signal,ADX_diff,ema12,ema26,eMACD1226,relative_macd_eMACD1226,eMACD1226_signal,eMACD1226_diff,relative_macd_diff,ema10,ema20,eMACD1020,eMACD1020_signal,eMACD1020_diff,relative_macd_eMACD1020,CxV,Sum CxV,Sum Volume,VWAP_5,VWAP_10,VWAP_20,ATRs_from_VWAP_5,ATRs_from_VWAP_10,ATRs_from_VWAP_20,Gap,Percent_Change,C-O,ATRs Traded_ex_gap,Relative_ATR,H-L,H-Cp,L-Cp,ATRs_Traded,u_band,l_band,relative_band_dist,VWAP 2026-07-13 00:00:00,ATRs_from_AVWAP_2026-07-13 00:00:00,VWAP 2026-07-06 00:00:00,ATRs_from_AVWAP_2026-07-06 00:00:00,VWAP 2026-06-18 00:00:00,ATRs_from_AVWAP_2026-06-18 00:00:00,VWAP 2026-01-01,hi_252,lo_252,bo_252,hi_50,lo_50,bo_50,hi_20,lo_20,bo_20,turtle50_20,hi_10,lo_10,bo_10,turtle20_10,hi_5,lo_5,bo_5,turtle10_5,sma50200,ema50200,ema2050,ema1020,ma_st,ma_mt,ma_lt,regime_crossing_ma_long_2050200,regime_crossing_ma_short_2050200,regime_crossing_ma_long_102050,regime_crossing_ma_short_102050,triple_ma2050200,Hi1,Lo1,Hi2,Lo2,Hi3,Lo3,Hi4,Lo4,rg,flr,clg,rg_ch,consolidation,acc_dist,w_acc_dist,c_over_under_20DMA,ep
Date,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2023-02-03,13.49,14.045,13.1700,13.22,2688518.0,13.5506,13.136,12.587,11.5555,11.4056,24.862650,2684592.40,1.001462,3.554221e+07,3.131005e+07,1.0200,0.781807,0.107443,0.809663,2.129042,2.320777,-14.891974,0.743916,61.133034,61.662320,-0.529286,14.525,10.085,70.608108,87.687688,-17.079580,38.444822,16.902256,19.232808,16.031401,3.201406,12.452794,11.853005,0.599790,0.767184,0.277661,0.322128,0.412030,12.630029,11.985298,0.644731,0.401081,0.243650,0.824668,3.554221e+07,NaN,NaN,13.329210,12.788417,11.662868,-0.139689,0.552033,1.991708,-4.933051,-6.835800,-2.001483,-0.946525,5.913820,0.8750,0.145,1.0200,1.304670,13.997126,9.113874,6.246108,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76.38,9.14,-1.0,14.710,9.140,-1.0,14.525,9.215,1.0,0,14.525,10.955,1.0,1,14.525,11.810,1.0,1,-1.0,-1.0,-1.0,1.0,11.5555,11.4056,24.862650,NaN,-1.0,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,98.58,NaN,NaN,NaN,NaN,NaN
2023-02-06,12.94,13.140,12.4925,12.60,3526764.0,12.6185,13.214,12.649,11.6995,11.3784,24.693950,2650942.50,1.330381,4.443723e+07,3.149082e+07,0.7275,0.777928,-0.789276,-0.062988,1.157562,1.570325,-15.546363,0.750719,55.551978,60.440252,-4.888274,14.525,10.085,56.644144,73.235736,-16.591592,35.414989,21.725085,20.562879,16.855306,3.707573,12.475441,11.908338,0.567104,0.728992,0.335550,0.231554,0.297654,12.624569,12.043841,0.580728,0.437010,0.143718,0.746506,4.443723e+07,NaN,NaN,13.360153,12.841318,11.879105,-0.977151,-0.310206,0.926686,-2.118003,-4.689864,-2.627512,-1.144065,6.174031,0.6475,0.080,0.7275,0.935177,14.022134,9.376866,5.971336,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76.38,9.14,-1.0,14.710,9.140,-1.0,14.525,9.920,1.0,0,14.525,11.005,1.0,1,14.525,12.175,1.0,1,-1.0,-1.0,-1.0,1.0,11.6995,11.3784,24.693950,NaN,-1.0,NaN,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-1.0,NaN,NaN,98.58,NaN,NaN,NaN,NaN,NaN
2023-02-07,12.58,12.930,12.3450,12.85,2901947.0,12.6500,13.244,12.744,11.8330,11.3534,24.542600,2632256.45,1.102456,3.729002e+07,3.168801e+07,0.5850,0.764147,-0.515607,0.138717,1.330895,1.958523,-15.301499,0.753404,57.246868,59.801575,-2.554707,14.525,10.085,62.274775,63.175676,-0.900901,32.840595,21.484012,20.789845,17.570677,3.219168,12.533066,11.978091,0.554975,0.726267,0.379435,0.175540,0.229720,12.665557,12.120618,0.544938,0.458596,0.086343,0.713133,3.729002e+07,NaN,NaN,13.358818,12.932456,12.038345,-0.665863,-0.107906,1.062171,-0.158730,1.984127,2.146264,-0.117778,5.946672,0.5850,0.330,0.2550,0.765559,14.094011,9.571989,5.917737,NaN,NaN,NaN,NaN,NaN,NaN,NaN,76.38,9.14,-1.0,14.710,9.140,-1.0,14.525,9.950,1.0,0,14.525,11.005,1.0,1,14.525,12.3